# state-of-the-art Chinese Font Generation Pipeline
This Colab notebook executes the pipeline using the cutting-edge **zi2zi-JiT** Diffusion Transformer. It sets up the environment, extracts characters from your handwriting, fine-tunes the engine (LoRA) on your style, and generates the final images.

## 1. Environment Setup
First, we will clone the repository, download zi2zi-JiT architecture and install the required dependencies.

In [ ]:
!git clone https://github.com/l2yao/chinese-font-generator.git
%cd chinese-font-generator
!git clone https://github.com/kaonashi-tyc/zi2zi-JiT.git
%cd zi2zi-JiT
!pip install torch torchvision torchaudio torch-fidelity --index-url https://download.pytorch.org/whl/cu118
!pip install -e .
!pip install opencv-python Pillow numpy fonttools svglib
!pip install gdown

## 1b. Install PaddleOCR (for character recognition/labeling)
PaddleOCR needs `paddlepaddle-gpu` matched to Colab's CUDA version. This installs the correct GPU build and PaddleOCR.

In [ ]:
# Remove any stale CPU-only paddlepaddle that Colab may have pre-installed
!pip uninstall -y paddlepaddle paddlepaddle-gpu 2>/dev/null

# Detect Colab CUDA version and install matching paddlepaddle-gpu
import subprocess, re
nvcc_out = subprocess.check_output(['nvcc', '--version']).decode()
cuda_match = re.search(r'release (\d+)\.(\d+)', nvcc_out)
cuda_major, cuda_minor = int(cuda_match.group(1)), int(cuda_match.group(2))
cu_tag = f'cu{cuda_major}{cuda_minor}'
print(f'Detected CUDA {cuda_major}.{cuda_minor} → installing paddlepaddle-gpu for {cu_tag}')

!python -m pip install paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/{cu_tag}/
!pip install "paddleocr>=3.4"

# Verify paddle can see the GPU
import paddle
paddle.utils.run_check()
print('PaddlePaddle GPU OK ✓')

## 2. Download Pretrained Weights
zi2zi-JiT provides base models. We will download the `JiT-B-16` weights.

In [ ]:
!mkdir -p models
!gdown 1chRW0YpKJ5Kh_5PFv1FMGehIpVixWO78 -O models/zi2zi-JiT-B-16.pth

## 3. Extract & Label Characters
Run the preprocessing script. It segments each character from the calligraphy images using OpenCV,
then automatically labels each crop with PaddleOCR (no manual `labels.txt` needed).

Output filenames: `<character>_<NNN>.png` (e.g. `佛_001.png`, `大_002.png`).

In [ ]:
!python ../src/preprocess/extract_grid.py --image_path ../data/train --output_dir data/sample_dataset/extracted

## 3a. Download a generic source font
zi2zi-JiT still needs a source glyph font to render the content side. Your raw handwriting images will be used as the style target, so the source font can be any readable Chinese font.

In [ ]:
!mkdir -p data/fonts
!wget -q -O data/fonts/SourceHanSerifSC-Regular.otf https://github.com/adobe-fonts/source-han-serif/raw/refs/heads/release/OTF/TraditionalChinese/SourceHanSerifTC-Regular.otf
!ls -lh data/fonts

## 3b. Build zi2zi Dataset
The extracted glyphs are already labeled by `extract_grid.py`. Now generate the paired dataset with `train/` and `test.npz`.

In [ ]:
# Generate the paired dataset for zi2zi-JiT using a source font.
!python scripts/generate_glyph_dataset.py \
    --source-font data/fonts/SourceHanSerifSC-Regular.otf \
    --glyph-dir data/sample_dataset/extracted \
    --output-dir data/sample_dataset \
    --train-count 200 \
    --font-name custom_handwriting


## 4. LoRA Fine-Tuning (Train your handwriting)
This runs the heavily optimized single-GPU LoRA training script.

In [ ]:
!python lora_single_gpu_finetune_jit.py \
    --data_path data/sample_dataset/train/ \
    --test_npz_path data/sample_dataset/test.npz \
    --output_dir run/lora_ft_sample_single/ \
    --base_checkpoint models/zi2zi-JiT-B-16.pth \
    --model JiT-B/16 \
    --batch_size 16

## 5. Generate and Vectorize!
Generate the stylized images and compile them into a TTF.

In [ ]:
# Generate characters
!python generate_chars.py --checkpoint run/lora_ft_sample_single/checkpoint-last.pth --output_dir run/generated_chars/ --sampling_method ab2

# Run our vectorizer
!python ../src/build_font/vectorize.py

from google.colab import files
files.download('custom_font.ttf')